In [4]:
import os
import re
import io
import time
import requests
import zipfile
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# ------------------------------------------------------------------
# 0. Paths + basic setup
# ------------------------------------------------------------------
os.makedirs("../data", exist_ok=True)
os.makedirs("../data/raw_gdelt", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# ------------------------------------------------------------------
# 1. Load company list (ticker + company_name)
# ------------------------------------------------------------------
company_df = pd.read_csv("../data/sp500_companies.csv")
# Expect columns: ticker, company_name (plus maybe CIK, sector, etc.)
company_df = company_df[['ticker', 'company_name']].dropna()
print("Loaded companies:", company_df.shape)

# ------------------------------------------------------------------
# 2. GDELT download helper
# ------------------------------------------------------------------
def download_gdelt_gkg(timestamp):
    """
    Download one GDELT GKG file.
    timestamp: 'YYYYMMDDHHMMSS', e.g. '20220103000000'
    Returns a pandas DataFrame or None.
    """
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    local_zip = f"../data/raw_gdelt/{timestamp}.gkg.csv.zip"

    # Simple caching: if file exists locally, read from disk
    if os.path.exists(local_zip):
        try:
            z = zipfile.ZipFile(local_zip)
            df = pd.read_csv(
                z.open(z.namelist()[0]),
                sep="\t",
                header=None,
                low_memory=False,
                dtype=str,
                encoding="latin-1"
            )
            return df
        except Exception as e:
            print(f"[{timestamp}] Local read error, re-downloading:", e)

    # Otherwise download
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            # 404/503 etc – just skip
            return None

        # cache zip locally
        with open(local_zip, "wb") as f:
            f.write(r.content)

        z = zipfile.ZipFile(io.BytesIO(r.content))
        df = pd.read_csv(
            z.open(z.namelist()[0]),
            sep="\t",
            header=None,
            low_memory=False,
            dtype=str,
            encoding="latin-1"
        )
        return df
    except Exception as e:
        print(f"[{timestamp}] Download error:", e)
        return None

# ------------------------------------------------------------------
# 3. Cleaning functions for URL + snippet
# ------------------------------------------------------------------
def clean_url_to_text(url):
    if not isinstance(url, str):
        return ""
    url = url.replace("https://", "").replace("http://", "")
    # remove domain
    url = re.sub(r'^[^/]+/', '', url)
    # replace separators with spaces
    url = re.sub(r'[-_/]', ' ', url)
    # keep only letters, numbers and spaces
    url = re.sub(r'[^a-zA-Z0-9 ]+', ' ', url)
    # collapse spaces
    url = re.sub(r'\s+', ' ', url)
    return url.strip().lower()

def clean_snippet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

# ------------------------------------------------------------------
# 4. Match GDELT rows to companies
# ------------------------------------------------------------------
def filter_company_news(gkg_df, company_df, date_str):
    """
    gkg_df: raw GDELT GKG df for a single timestamp
    company_df: DataFrame with columns ['ticker', 'company_name']
    date_str: 'YYYYMMDD'
    Returns: list of dicts with fields: date, ticker, url, alt_url, snippet, headline
    """
    results = []
    if gkg_df is None or gkg_df.empty:
        return results

    # Column indices based on your inspection:
    # 1: (second column) – date / entity text (we still include for matching)
    # 4: main URL
    # 24: snippet text
    # 26: alternate URL (PAGE_LINKS / PAGE_ALTURL_AMP)
    col1    = gkg_df[1].astype(str).str.lower()
    url_col = gkg_df[4].astype(str).str.lower()
    snip    = gkg_df[24].astype(str).str.lower()
    alt_col = gkg_df[26].astype(str).str.lower()

    for _, c in company_df.iterrows():
        ticker  = c['ticker']
        company = str(c['company_name']).lower()
        if not company:
            continue
        key = company.split()[0]  # first word: Apple, Microsoft, Amazon, ...

        mask = (
            col1.str.contains(key,   na=False) |
            url_col.str.contains(key, na=False) |
            snip.str.contains(key,    na=False) |
            alt_col.str.contains(key, na=False)
        )

        matched = gkg_df[mask]
        if matched.empty:
            continue

        for _, row in matched.iterrows():
            results.append({
                "date": date_str,
                "ticker": ticker,
                "url": row[4],
                "alt_url": row[26],
                "snippet": row[24],
                "headline": row[4]  # URL as base for headline
            })

    return results

# ------------------------------------------------------------------
# 5. FinBERT on GPU
# ------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("FinBERT running on:", device)

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
model.eval()

def finbert_gpu(texts, batch_size=64):
    """
    texts: list of strings
    returns: np.ndarray [n_texts, 3] with probs for [neg, neu, pos]
    """
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

# ------------------------------------------------------------------
# 6. Main loop: build daily news sentiment
# ------------------------------------------------------------------

# Configure date range (you can shrink this first for testing)
START_DATE = "2021-01-01"
END_DATE   = "2021-01-02"

all_daily_dfs = []

date_range = pd.date_range(START_DATE, END_DATE, freq="D")
print(f"Total days to process: {len(date_range)}")

start_time_global = time.time()

for idx, dt in enumerate(date_range, start=1):
    date_str = dt.strftime("%Y%m%d")
    display_date = dt.strftime("%Y-%m-%d")
    daily_rows = []

    day_start_time = time.time()

    # Loop over 24 hours
    for hour in range(24):
        ts = f"{date_str}{hour:02d}0000"  # HH:00:00
        gkg = download_gdelt_gkg(ts)
        if gkg is None:
            continue
        rows = filter_company_news(gkg, company_df, date_str)
        daily_rows.extend(rows)

    if not daily_rows:
        # No matched news for any ticker on this day
        print(f"[{idx}/{len(date_range)}] {display_date}: no matched news")
        continue

    news_raw = pd.DataFrame(daily_rows).drop_duplicates(subset=["date", "ticker", "url"])
    # Clean and build combined text
    news_raw['headline_text'] = news_raw['url'].apply(clean_url_to_text)
    news_raw['snippet_text']  = news_raw['snippet'].apply(clean_snippet)

    news_raw['combined_text'] = (
        news_raw['headline_text'] + " " + news_raw['snippet_text']
    ).str.strip()

    # Filter out completely empty combined_text
    news_raw = news_raw[news_raw['combined_text'] != ""]
    if news_raw.empty:
        print(f"[{idx}/{len(date_range)}] {display_date}: all texts empty after cleaning")
        continue

    # Run FinBERT
    probs = finbert_gpu(news_raw['combined_text'].tolist(), batch_size=64)
    news_raw['neg'] = probs[:, 0]
    news_raw['neu'] = probs[:, 1]
    news_raw['pos'] = probs[:, 2]

    # Aggregate to daily ticker-level features
    daily_news = (
        news_raw.groupby(['date', 'ticker'])
        .agg(
            sent_mean=('pos', 'mean'),
            sent_vol=('pos', 'std'),
            pos_ratio=('pos', lambda x: (x > 0.5).mean()),
            neg_ratio=('neg', lambda x: (x > 0.5).mean()),
            neu_ratio=('neu', lambda x: (x > 0.5).mean()),
            news_count=('combined_text', 'count')
        )
        .fillna(0)
        .reset_index()
    )

    all_daily_dfs.append(daily_news)

    day_elapsed = time.time() - day_start_time
    print(
        f"[{idx}/{len(date_range)}] {display_date}: "
        f"{len(news_raw)} articles → {daily_news.shape[0]} ticker-days "
        f"in {day_elapsed:.1f}s"
    )

# ------------------------------------------------------------------
# 7. Concatenate all days + save
# ------------------------------------------------------------------
if all_daily_dfs:
    news_daily = pd.concat(all_daily_dfs, ignore_index=True)
    # Make date a proper datetime if you like:
    # news_daily['date'] = pd.to_datetime(news_daily['date'], format="%Y%m%d")
    out_path = "../data/processed/news_daily.parquet"
    news_daily.to_parquet(out_path, index=False)
    total_elapsed = time.time() - start_time_global
    print(f"\nSaved {news_daily.shape[0]} rows to {out_path}")
    print(f"Total elapsed time: {total_elapsed/60:.1f} minutes")
else:
    print("No news data was generated – check matching logic or date range.")


Loaded companies: (503, 2)
FinBERT running on: cuda
Total days to process: 2


KeyboardInterrupt: 

In [ ]:
news_daily

NameError: name 'news_daily' is not defined

In [16]:
import time, random

time.sleep(random.uniform(0.5, 1.5))


In [17]:
import requests
import pandas as pd
from datetime import datetime

def fetch_yahoo_news(ticker, verbose=True):
    """
    Fetch Yahoo Finance news using the reliable query1.finance.yahoo.com search endpoint.
    Returns a clean DataFrame with title, summary, link, datetime, tickers.
    """
    try:
        url = f"https://query1.finance.yahoo.com/v1/finance/search?q={ticker}"
        # prevent throttle
        time.sleep(random.uniform(0.5, 1.5))
        r = requests.get(url, timeout=10)

        if r.status_code != 200:
            print(f"Failed to fetch news for {ticker}: HTTP {r.status_code}")
            return pd.DataFrame()

        data = r.json()

        if "news" not in data:
            print(f"No news found for {ticker}")
            return pd.DataFrame()

        items = data["news"]
        if not isinstance(items, list):
            print(f"No valid news list for {ticker}")
            return pd.DataFrame()

        # Extract useful fields
        rows = []
        for item in items:
            rows.append({
                "title": item.get("title"),
                "summary": item.get("summary"),
                "publisher": item.get("publisher"),
                "link": item.get("link"),
                "published_time": item.get("publishTime"),
                "datetime": datetime.fromtimestamp(item.get("publishTime")) if item.get("publishTime") else None,
                "tickers": item.get("tickerSymbols"),
                "relatedTickers": item.get("relatedTickers"),
                "type": item.get("type")
            })

        df = pd.DataFrame(rows)
        df["date"] = df["datetime"].dt.date

        if verbose:
            print(f"Fetched {len(df)} news articles for {ticker}")
            display(df.head())

        return df

    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")
        return pd.DataFrame()


In [18]:
msft_news = fetch_yahoo_news("MSFT")
msft_news.head()


Failed to fetch news for MSFT: HTTP 429


""


In [19]:
pip install feedparser


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6089 sha256=9b70502db61443f3bdd9be4dfc055fb647c7c99a98cae72d20f16864a0488d81
  Stored in directory: /orcd/home/002/zeyuzh/.cache/pip/wheels/3d/4d/ef/37cdccc18d6fd7e0dd7817dcdf9146d4d6789c32a227a28134
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]2 [feedparser]
Note: you may need to restart the kernel to use updated packages.


In [20]:
import feedparser
import pandas as pd
from datetime import datetime

def fetch_yahoo_news_rss(ticker, verbose=True):
    """
    Fetch Yahoo Finance news using the RSS endpoint.
    This method NEVER hits 429, is stable, and returns ticker-linked headlines.
    """
    try:
        url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US"
        feed = feedparser.parse(url)

        if not feed.entries:
            if verbose:
                print(f"No RSS news found for {ticker}")
            return pd.DataFrame()

        rows = []
        for entry in feed.entries:
            rows.append({
                "title": entry.get("title"),
                "summary": entry.get("summary"),
                "link": entry.get("link"),
                "published": entry.get("published"),
                "datetime": datetime(*entry.published_parsed[:6]) if entry.get("published_parsed") else None,
                "date": datetime(*entry.published_parsed[:6]).date() if entry.get("published_parsed") else None,
                "tickers": [ticker]  # RSS feed is already ticker-filtered
            })

        df = pd.DataFrame(rows)

        if verbose:
            print(f"Fetched {len(df)} RSS news articles for {ticker}")
            display(df.head())

        return df

    except Exception as e:
        print("Error fetching RSS news:", e)
        return pd.DataFrame()


In [21]:
msft_news = fetch_yahoo_news_rss("MSFT")
msft_news.head()


Fetched 20 RSS news articles for MSFT


,title,summary,link,published,datetime,date,tickers
0,2 Artificial Intelligence (AI) Stocks You Can ...,The past 10 years have been kind to these two ...,https://www.fool.com/investing/2025/11/22/2-ar...,"Sun, 23 Nov 2025 01:06:00 +0000",2025-11-23 01:06:00,2025-11-23,[MSFT]
1,Top Stocks to Double Up on Right Now,"If owning some is good, holding more may be ev...",https://www.fool.com/investing/2025/11/22/top-...,"Sun, 23 Nov 2025 00:18:00 +0000",2025-11-23 00:18:00,2025-11-23,[MSFT]
2,Can Lumen Technologies’ (LUMN) New AI Partners...,"Earlier this week, Lumen Technologies launched...",https://finance.yahoo.com/news/lumen-technolog...,"Sat, 22 Nov 2025 22:13:50 +0000",2025-11-22 22:13:50,2025-11-22,[MSFT]
3,AI Bubble Fears Spark a Sell-Off: 1 Stock to B...,Time to buy? It depends on what you're conside...,https://www.fool.com/investing/2025/11/22/ai-b...,"Sat, 22 Nov 2025 20:41:00 +0000",2025-11-22 20:41:00,2025-11-22,[MSFT]
4,Billionaire Stanley Druckenmiller Just Bought ...,"Druckenmiller opened new positions in Amazon, ...",https://www.fool.com/investing/2025/11/22/bill...,"Sat, 22 Nov 2025 20:17:00 +0000",2025-11-22 20:17:00,2025-11-22,[MSFT]


,title,summary,link,published,datetime,date,tickers
0,2 Artificial Intelligence (AI) Stocks You Can ...,The past 10 years have been kind to these two ...,https://www.fool.com/investing/2025/11/22/2-ar...,"Sun, 23 Nov 2025 01:06:00 +0000",2025-11-23 01:06:00,2025-11-23,[MSFT]
1,Top Stocks to Double Up on Right Now,"If owning some is good, holding more may be ev...",https://www.fool.com/investing/2025/11/22/top-...,"Sun, 23 Nov 2025 00:18:00 +0000",2025-11-23 00:18:00,2025-11-23,[MSFT]
2,Can Lumen Technologies’ (LUMN) New AI Partners...,"Earlier this week, Lumen Technologies launched...",https://finance.yahoo.com/news/lumen-technolog...,"Sat, 22 Nov 2025 22:13:50 +0000",2025-11-22 22:13:50,2025-11-22,[MSFT]
3,AI Bubble Fears Spark a Sell-Off: 1 Stock to B...,Time to buy? It depends on what you're conside...,https://www.fool.com/investing/2025/11/22/ai-b...,"Sat, 22 Nov 2025 20:41:00 +0000",2025-11-22 20:41:00,2025-11-22,[MSFT]
4,Billionaire Stanley Druckenmiller Just Bought ...,"Druckenmiller opened new positions in Amazon, ...",https://www.fool.com/investing/2025/11/22/bill...,"Sat, 22 Nov 2025 20:17:00 +0000",2025-11-22 20:17:00,2025-11-22,[MSFT]


In [26]:
# first row of summary column
msft_news["summary"].iloc[0]

'The past 10 years have been kind to these two tech leaders. Things could look much the same through the next decade.'

In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_finviz_news(ticker):
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get(url, headers=headers)
    
    if resp.status_code != 200:
        print(f"Error fetching {ticker}: HTTP {resp.status_code}")
        return pd.DataFrame([])
    
    soup = BeautifulSoup(resp.text, "html.parser")
    
    # New correct ID for FinViz news table
    news_table = soup.find("table", {"id": "news-table"})
    if news_table is None:
        print(f"No news table found for {ticker}")
        return pd.DataFrame([])
    
    rows = news_table.find_all("tr")
    data = []
    last_date = None

    for row in rows:
        # Skip rows without a link (these are separators or ads)
        a_tag = row.find("a")
        if a_tag is None:
            continue

        title = a_tag.text.strip()
        
        # Sometimes the <td> has date + time OR only time
        date_time = row.td.text.strip().split()

        if len(date_time) == 2:  # e.g. "Nov-23-24 09:15"
            last_date = date_time[0]
            time_only = date_time[1]
        else:                    # e.g. "09:15" (same date as previous)
            time_only = date_time[0]

        # Source is inside <span> if available
        span_tag = row.find("span")
        source = span_tag.text.strip() if span_tag else ""

        data.append({
            "ticker": ticker,
            "date": last_date,
            "time": time_only,
            "title": title,
            "source": source
        })

    return pd.DataFrame(data)


In [29]:
df = get_finviz_news("AAPL")
df.head()


,ticker,date,time,title,source
0,AAPL,Today,01:30PM,Where Will Apple Stock Be in 5 Years?,(Motley Fool)
1,AAPL,Today,12:26PM,Meta wants to get into the electricity trading...,(TechCrunch)
2,AAPL,Today,10:34AM,"Apples presumptive future CEO, John Ternus, ha...",(Yahoo Finance)
3,AAPL,Today,08:32AM,Stock Market Week Ahead: A Trillion-Dollar Sho...,(Investor's Business Daily)
4,AAPL,Today,07:30AM,These two 'Magnificent Seven' stocks could be ...,(MarketWatch)


In [32]:
from datasets import load_dataset
ds = load_dataset("Zihan1004/FNSPID", split="news")  # or check exact name
print(ds.column_names)
print(ds.select(range(5)))


Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 1400000 examples [00:05, 241492.15 examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [33]:
from datasets import get_dataset_split_names

splits = get_dataset_split_names("Zihan1004/FNSPID")
print(splits)


Repo card metadata block was not found. Setting CardData to empty.


['train']


In [1]:
import pandas as pd

url1 = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/All_external.csv"
url2 = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv"

df1 = pd.read_csv(url1, low_memory=False)
# df2 = pd.read_csv(url2, low_memory=False)

print(df1.head())



                      Date                                      Article_title  \
0  2020-06-05 06:30:54 UTC            Stocks That Hit 52-Week Highs On Friday   
1  2020-06-03 06:45:20 UTC         Stocks That Hit 52-Week Highs On Wednesday   
2  2020-05-26 00:30:07 UTC                      71 Biggest Movers From Friday   
3  2020-05-22 08:45:06 UTC       46 Stocks Moving In Friday's Mid-Day Session   
4  2020-05-22 07:38:59 UTC  B of A Securities Maintains Neutral on Agilent...   

  Stock_symbol                                                Url  \
0            A  https://www.benzinga.com/news/20/06/16190091/s...   
1            A  https://www.benzinga.com/news/20/06/16170189/s...   
2            A  https://www.benzinga.com/news/20/05/16103463/7...   
3            A  https://www.benzinga.com/news/20/05/16095921/4...   
4            A  https://www.benzinga.com/news/20/05/16095304/b...   

           Publisher Author Article  Lsa_summary  Luhn_summary  \
0  Benzinga Insights    NaN     

In [3]:
df1.head(20)
df1.sample(20)
df1[df1["Stock_symbol"]=="AAPL"].head()

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
6680,2020-06-10 07:33:26 UTC,Tech Stocks And FAANGS Strong Again To Start D...,AAPL,https://www.benzinga.com/government/20/06/1622...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN
6681,2020-06-10 04:14:08 UTC,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
6682,2020-06-10 03:53:47 UTC,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",AAPL,https://www.benzinga.com/short-sellers/20/06/1...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
6683,2020-06-10 03:19:25 UTC,"Deutsche Bank Maintains Buy on Apple, Raises P...",AAPL,https://www.benzinga.com/news/20/06/16219873/d...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
6684,2020-06-10 02:27:11 UTC,Apple To Let Users Trade In Their Mac Computer...,AAPL,https://www.benzinga.com/news/20/06/16218697/a...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import pandas as pd

df1["Date"] = pd.to_datetime(df1["Date"], errors="coerce")
df1["Date"].max()

Timestamp('2020-06-11 13:12:35+0000', tz='UTC')

In [5]:
df2 = pd.read_csv("../data/nasdaq_exteral_data.csv", nrows=200000)
df2["Date"] = pd.to_datetime(df2["Date"], errors="coerce")
df2["Date"].max()


Timestamp('2024-01-09 00:00:00+0000', tz='UTC')

In [6]:
df2

,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,0.0,2023-12-16 23:00:00+00:00,Interesting A Put And Call Options For August ...,A,https://www.nasdaq.com/articles/interesting-a-...,NaN,NaN,"Investors in Agilent Technologies, Inc. (Symbo...",Because the $125.00 strike represents an appro...,The current analytical data (including greeks ...,Below is a chart showing the trailing twelve m...,"At Stock Options Channel, our YieldBoost formu..."
1,1.0,2023-12-12 00:00:00+00:00,Wolfe Research Initiates Coverage of Agilent T...,A,https://www.nasdaq.com/articles/wolfe-research...,NaN,NaN,"Fintel reports that on December 13, 2023, Wolf...","Fintel reports that on December 13, 2023, Wolf...","T. Rowe Price Investment Management holds 10,1...",Agilent Technologies Declares $0.24 Dividend O...,The projected annual revenue for Agilent Techn...
2,2.0,2023-12-12 00:00:00+00:00,Agilent Technologies Reaches Analyst Target Price,A,https://www.nasdaq.com/articles/agilent-techno...,NaN,NaN,"In recent trading, shares of Agilent Technolog...","In recent trading, shares of Agilent Technolog...","In recent trading, shares of Agilent Technolog...",When a stock reaches the target an analyst has...,When a stock reaches the target an analyst has...
3,3.0,2023-12-07 00:00:00+00:00,Agilent (A) Enhances BioTek Cytation C10 With ...,A,https://www.nasdaq.com/articles/agilent-a-enha...,NaN,NaN,Agilent Technologies A is enhancing its BioTek...,"Per a Grand View Research report, the global m...","Notably, Agilent enhanced the BioTek Cytation ...","Agilent Technologies, Inc. Price and Consensus...","Notably, Agilent enhanced the BioTek Cytation ..."
4,4.0,2023-12-07 00:00:00+00:00,"Pre-Market Most Active for Dec 7, 2023 : SQQQ,...",A,https://www.nasdaq.com/articles/pre-market-mos...,NaN,NaN,The NASDAQ 100 Pre-Market Indicator is up 70.2...,ProShares UltraPro Short QQQ (SQQQ) is -0.15 a...,"As reported by Zacks, the current mean recomme...","The total Pre-Market volume is currently 39,23...",The NASDAQ 100 Pre-Market Indicator is up 70.2...
...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199995.0,2015-07-28 00:00:00+00:00,"Plains All American Pipeline, L.P. (PAA) Ex-Di...",AMZA,https://www.nasdaq.com/articles/plains-all-ame...,NaN,NaN,"Plains All American Pipeline, L.P. ( PAA ) wil...",The following ETF(s) have PAA as a top-10 hold...,The views and opinions expressed herein are th...,The following ETF(s) have PAA as a top-10 hold...,The following ETF(s) have PAA as a top-10 hold...
199996,199996.0,2015-07-14 00:00:00+00:00,MLP ETFs in Focus after MLPX -- MarkWest Deal ...,AMZA,https://www.nasdaq.com/articles/mlp-etfs-in-fo...,NaN,NaN,Refinery and pipeline company Marathon Petrole...,InfraCap MLP ETF ( AMZA ) The actively managed...,The fund currently manages an asset base of $1...,InfraCap MLP ETF ( AMZA ) The actively managed...,Click to get this free report MARATHON PETROL ...
199997,199997.0,2015-05-12 00:00:00+00:00,"Spectra Energy Partners, LP (SEP) Ex-Dividend ...",AMZA,https://www.nasdaq.com/articles/spectra-energy...,NaN,NaN,"Spectra Energy Partners, LP ( SEP ) will begin...",The following ETF(s) have SEP as a top-10 hold...,The following ETF(s) have SEP as a top-10 hold...,The following ETF(s) have SEP as a top-10 hold...,The following ETF(s) have SEP as a top-10 hold...
199998,199998.0,2015-05-08 00:00:00+00:00,Spectra Energy Corp (SE) Ex-Dividend Date Sche...,AMZA,https://www.nasdaq.com/articles/spectra-energy...,NaN,NaN,Spectra Energy Corp ( SE ) will begin trading ...,The following ETF(s) have SE as a top-10 holdi...,The following ETF(s) have SE as a top-10 holdi...,The following ETF(s) have SE as a top-10 holdi...,The following ETF(s) have SE as a top-10 holdi...
